# 04 Apply a trained model to new data

Once a codebook is trained, `predict_codes` assigns the same cell and neighborhood codes to a new sample **without retraining**. This is how you annotate a held-out slide with an existing codebook.

We train on three of the four MERFISH retina samples and apply the codebook to the held-out fourth. The neighbor-graph arguments passed to `predict_codes` must match the ones used at training time.

In [ ]:
import numpy as np
import nicheverse as nv
from nicheverse import ModelConfig, TrainConfig, predict_codes
import os
MERFISH = os.path.join('..', 'examples', 'data', 'merfish_retina.h5ad')
adata = nv.read_spatial(MERFISH, sample_col='sample_id')
samples = sorted(adata.obs['sample_id'].astype(str).unique())
held_out = samples[-1]
train_samples = samples[:-1]
print('train on:', train_samples, '  held out:', held_out)

In [ ]:
is_train = adata.obs['sample_id'].astype(str).isin(train_samples)
train_ad = adata[is_train].copy()
new_ad = adata[~is_train].copy()
print('train cells:', train_ad.n_obs, ' held-out cells:', new_ad.n_obs)

## Train the codebook

The graph settings are captured once and reused for both train and predict so the neighborhood codes stay comparable.

In [ ]:
# b32k reference defaults (encoder mlp_deep, vq, knn_radius@50um, k=20, weighted_mean, seed 9);
# only num_epochs (300 -> 30) and batch_size (32768 -> 2048) are shrunk for the demo.
GRAPH = dict(k_neighbors=20, neighborhood_aggregation='weighted_mean',
             spatial_graph='knn_radius', radius=50.0)
mc = ModelConfig(input_dim=train_ad.n_vars, gene_names=tuple(train_ad.var_names.astype(str)),
                 encoder_type='mlp_deep', cell_num_embeddings=256, neighborhood_num_embeddings=32)
tc = TrainConfig(num_epochs=30, batch_size=2048, save_best=False, seed=9, **GRAPH)
CKPT = 'runs/apply_demo'
model, train_ad = nv.train_model(train_ad, CKPT, model_config=mc, train_config=tc, sample_col='sample_id')
print('trained; codes used:', train_ad.obs['cell_codebook_idx'].nunique(), '/ 256')

## Assign codes to the held-out sample

`predict_codes` loads the checkpoint, aligns the gene panel, builds the neighbor graph the same way, and writes `cell_codebook_idx` / `neighborhood_codebook_idx` plus the embeddings onto the new AnnData. It never updates the codebook.

In [ ]:
ckpt_pt = f'{CKPT}/hierarchical_vqvae_checkpoint.pt'
coded = predict_codes(new_ad, ckpt_pt, sample_col='sample_id', **GRAPH,
                      output_path='runs/held_out_annotated.h5ad')
c = coded.obs['cell_codebook_idx'].to_numpy()
print('held-out cells coded:', coded.n_obs)
print('distinct cell codes on held-out sample:', len(np.unique(c)))
print('niches on held-out sample:', coded.obs['neighborhood_codebook_idx'].nunique())
assert 'X_cell_embedding' in coded.obsm
print('embeddings attached:', coded.obsm['X_cell_embedding'].shape)

The held-out sample now carries the **same** codes as the training cohort, so a cell assigned code 42 in training and code 42 here refer to the same learned state. That is what makes the codebook a shared, transferable annotation across samples and studies.